In [ ]:
# %% [Part 0] 写配置文件 —— 只需运行一次，改了参数才需要重新运行这个cell
from pathlib import Path

PROJECT_ROOT = Path("/Volumes/Elements/DEPA_downstream")
yaml_path = PROJECT_ROOT / "configs" / "pre_t.yaml"

yaml_content = """cnmf_dir: "/Volumes/Elements/data/ALL_Pre-T/cNMF_output"
run_name: "PreT_DEAPA_programs"
K: 8
density_threshold: "0_01"

h5ad_path: "/Volumes/Elements/data/ALL_Pre-T/cNMF_output/PreT_DEAPA_counts.h5ad"
feature_anno_path: "/Volumes/Elements/data/ALL_Pre-T/cNMF_input/cnmf_feature_anno.csv"

wpsi_matrix_path: "/Volumes/Elements/data/ALL_Pre-T/PSI_results/movAPA_wPSI_matrix_df.rds"
psi_matrix_path: "/Volumes/Elements/data/ALL_Pre-T/PSI_results/movAPA_PSI_matrix_df.rds"
depa_feature_source_path: "/Volumes/Elements/data/ALL_Pre-T/cNMF_input/DEPA_feature_source.csv"
gene_symbol_map_path: "/Volumes/Elements/DEPA_downstream/configs/pre_t_gene_symbol_map.csv"

meta_columns:
  sample: "sample"
  celltype: "cell_type_final"
  umap1: "UMAP_1"
  umap2: "UMAP_2"

contrasts:
  - name: "disease"
    column: "group"
    type: "pairwise"
    group_a: "PBMMC"
    group_b: "PreT_ALL"
    category: "disease"

  - name: "celltype_identity"
    column: "cell_type_final"
    type: "auto_one_vs_rest_all"

z_threshold: 1.5
background_scope: "pac_universe"
min_cells_per_group: 20
enrichr_databases: [GO_Biological_Process_2023, KEGG_2021_Human, MSigDB_Hallmark_2020, Reactome_2022]
output_root: "/Volumes/Elements/DEPA_downstream/outputs/pre_T/"
"""
yaml_path.write_text(yaml_content, encoding="utf-8")
print(f"写入完成: {yaml_path}")

# 加个保护：文件已存在就不覆盖，改配置时手动删掉旧文件或直接编辑yaml本身
if yaml_path.exists():
    print(f"配置文件已存在，跳过写入: {yaml_path}\n"
          f"（如需更新配置，直接编辑该yaml文件，或删除后重新运行本cell）")
else:
    yaml_path.write_text(yaml_content, encoding="utf-8")
    print(f"写入完成: {yaml_path}")

In [ ]:
# %% [Part 0] 加载依赖与数据 —— 每次重启kernel后运行一次即可，
# 后面三层分析都复用这里的 data / CONFIG，不用重复加载
%load_ext autoreload
%autoreload 2

import yaml
import pandas as pd
from pathlib import Path
from depa_downstream import (
    load_cnmf_outputs,
    compute_program_direction, plot_program_direction_summary,
    compute_leading_genes, compare_within_celltype, summarize_leading_gene_within_celltype,   # 新增后两个
    compute_contrast_association, plot_association_heatmap, plot_program_umap_grid,
    compute_program_gene_sets, compute_jaccard_matrix, select_program_markers,
    resolve_module_labels, plot_jaccard_heatmap,
    run_enrichment, plot_enrichment_bubble,
    plot_leading_genes_bar, plot_gene_wpsi_by_group,           # 之前加过的也顺带确认在里面
    plot_within_celltype_comparison, plot_within_celltype_effect_summary,   # 这次新增
)

CONFIG = yaml.safe_load(open(PROJECT_ROOT / "configs" / "pre_t.yaml"))
Path(CONFIG["output_root"]).mkdir(parents=True, exist_ok=True)
print("gene_symbol_map_path:", CONFIG["gene_symbol_map_path"])
CONFIG

data = load_cnmf_outputs(CONFIG)
print("programs:", data["programs"])
print("meta细胞数:", len(data["meta"]))
print("cell_type_final取值:", data["meta"]["cell_type_final"].unique())


In [ ]:
# %% [Layer 1] program 的 wPSI 方向（Proximal / Distal）
program_direction = compute_program_direction(data, CONFIG)
plot_program_direction_summary(program_direction, CONFIG, CONFIG["output_root"] + "layer1")
program_direction


In [ ]:
# %% [Layer 1 深入] leading genes：谁在驱动每个program的方向
leading_genes = compute_leading_genes(data, CONFIG, program_direction, top_n=15)
leading_genes["P02"]


In [ ]:
# %% [Layer 1 深入] leading genes条形图（对所有完成检验的program批量画）
for prog in data["programs"]:
    direction = program_direction.loc[prog, "direction"]
    if direction != "NA":
        plot_leading_genes_bar(leading_genes, prog, CONFIG["output_root"] + "layer1", direction_label=direction)

In [ ]:
# %% [Layer 1 深入] 挑P02的top leading gene，看它在disease分组下的具体wPSI分布
top_gene = leading_genes["P02"].iloc[0]
plot_gene_wpsi_by_group(
    top_gene["gene_id"], data, CONFIG,
    groupby_col=CONFIG["contrasts"][0]["column"],  # "group" -> PBMMC/PreT_ALL
    out_dir=CONFIG["output_root"] + "layer1",
    gene_symbol=top_gene["symbol"],
)

In [ ]:
# %% [Layer 1 深入] 同一个基因也可以按细胞类型看
plot_gene_wpsi_by_group(
    top_gene["gene_id"], data, CONFIG,
    groupby_col=CONFIG["meta_columns"]["celltype"],
    out_dir=CONFIG["output_root"] + "layer1",
    gene_symbol=top_gene["symbol"],
)

In [ ]:
# %% [Layer 1 深入] 组内比较：限定在 Proliferating T-ALL 内部，
# 比较PBMMC来源(正常增殖细胞) vs PreT_ALL来源(恶性增殖细胞)
within_result = compare_within_celltype(
    data, CONFIG,
    celltype="Proliferating T-ALL",
    disease_column=CONFIG["contrasts"][0]["column"],   # "group"
    group_a="PBMMC", group_b="PreT_ALL",
    gene_id=top_gene["gene_id"],   # 复用之前leading gene里选的CENPM
    program="P02",
)
within_result

In [ ]:
# %% [Layer 1 深入] 组内比较画图 —— wPSI层面
wpsi = data["wpsi"]
meta = data["meta"]
celltype_col = CONFIG["meta_columns"]["celltype"]
disease_col = CONFIG["contrasts"][0]["column"]

mask = meta[celltype_col] == "Proliferating T-ALL"
sub_meta = meta.loc[mask]
common_cells = wpsi.columns.intersection(sub_meta.index)
vals = wpsi.loc[top_gene["gene_id"], common_cells]
groups = sub_meta.loc[common_cells, disease_col]

plot_within_celltype_comparison(
    vals[groups == "PBMMC"].dropna().values,
    vals[groups == "PreT_ALL"].dropna().values,
    label_a="PBMMC\n(normal proliferating)",
    label_b="PreT_ALL\n(malignant proliferating)",
    ylabel="wPSI (per cell)",
    title=f"{top_gene['symbol']} wPSI — within Proliferating T-ALL cluster",
    out_path=CONFIG["output_root"] + f"layer1/layer1_within_celltype_wpsi_{top_gene['symbol']}.png",
)

In [ ]:
# %% [Layer 1 深入] 组内比较画图 —— P02 usage层面
usage = data["usage"]
common_cells = usage.index.intersection(sub_meta.index)
u_vals = usage.loc[common_cells, "P02"]
u_groups = sub_meta.loc[common_cells, disease_col]

plot_within_celltype_comparison(
    u_vals[u_groups == "PBMMC"].dropna().values,
    u_vals[u_groups == "PreT_ALL"].dropna().values,
    label_a="PBMMC\n(normal proliferating)",
    label_b="PreT_ALL\n(malignant proliferating)",
    ylabel="P02 usage",
    title="P02 usage — within Proliferating T-ALL cluster",
    out_path=CONFIG["output_root"] + "layer1/layer1_within_celltype_usage_P02.png",
)

In [ ]:
# 1. 看两组的检出率是不是本身就不一样
print("PBMMC检出率:", 324/733, " PreT_ALL检出率:", 978/2287)

# 2. 效应量：非零细胞比例，比中位数更能反映真实差异在哪
a_vals = vals[groups == "PBMMC"].dropna()
b_vals = vals[groups == "PreT_ALL"].dropna()
print("PBMMC中 wPSI>0 的比例:", (a_vals > 0).mean())
print("PreT_ALL中 wPSI>0 的比例:", (b_vals > 0).mean())
print("PBMMC均值:", a_vals.mean(), " PreT_ALL均值:", b_vals.mean())

In [ ]:
# 系统性跑一遍P02所有leading gene的组内对照
summary_rows = []
for _, row in leading_genes["P02"].iterrows():
    res = compare_within_celltype(
        data, CONFIG,
        celltype="Proliferating T-ALL",
        disease_column=CONFIG["contrasts"][0]["column"],
        group_a="PBMMC", group_b="PreT_ALL",
        gene_id=row["gene_id"],
    )
    if "wpsi" in res:
        summary_rows.append({
            "symbol": row["symbol"],
            "median_a": res["wpsi"]["median_a"],
            "median_b": res["wpsi"]["median_b"],
            "pval": res["wpsi"]["pval"],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df["direction_consistent"] = summary_df["median_b"] > summary_df["median_a"]  # 是否和P02.usage同方向
summary_df

In [ ]:
def compare_within_celltype_effect_size(gene_id, data, config, celltype, disease_column, group_a, group_b):
    """用非零比例+均值代替median比较，避免稀疏性导致的误判（同CENPM诊断逻辑）"""
    wpsi = data["wpsi"]
    meta = data["meta"]
    celltype_col = config["meta_columns"]["celltype"]
    mask = meta[celltype_col] == celltype
    sub_meta = meta.loc[mask]
    common_cells = wpsi.columns.intersection(sub_meta.index)
    vals = wpsi.loc[gene_id, common_cells]
    groups = sub_meta.loc[common_cells, disease_column]
    a_vals = vals[groups == group_a].dropna()
    b_vals = vals[groups == group_b].dropna()
    if len(a_vals) < 20 or len(b_vals) < 20:
        return None
    from scipy.stats import mannwhitneyu
    _, pval = mannwhitneyu(a_vals, b_vals, alternative="two-sided")
    return {
        "nonzero_frac_a": (a_vals > 0).mean(), "nonzero_frac_b": (b_vals > 0).mean(),
        "mean_a": a_vals.mean(), "mean_b": b_vals.mean(),
        "pval": pval,
    }

rows = []
for _, row in leading_genes["P02"].iterrows():
    res = compare_within_celltype_effect_size(
        row["gene_id"], data, CONFIG,
        celltype="Proliferating T-ALL",
        disease_column=CONFIG["contrasts"][0]["column"],
        group_a="PBMMC", group_b="PreT_ALL",
    )
    if res:
        rows.append({"symbol": row["symbol"], **res})

effect_df = pd.DataFrame(rows)
effect_df["mean_diff"] = effect_df["mean_b"] - effect_df["mean_a"]
effect_df["direction_consistent"] = effect_df["mean_diff"] > 0
effect_df["significant"] = effect_df["pval"] < 0.05
effect_df.sort_values("pval")

In [ ]:
effect_df["abs_mean_diff"] = effect_df["mean_diff"].abs()
effect_df_sorted = effect_df.sort_values("abs_mean_diff", ascending=False)
effect_df_sorted

In [ ]:
pd.set_option("display.max_rows", 20)
effect_df_sorted

In [ ]:
threshold = 0.01
def classify(row):
    if row["pval"] >= 0.05 or row["abs_mean_diff"] < threshold:
        return "no_meaningful_diff"
    return "supports_P02" if row["mean_diff"] > 0 else "opposes_P02"

effect_df_sorted["category"] = effect_df_sorted.apply(classify, axis=1)
effect_df_sorted["category"].value_counts()

In [ ]:
meaningful = effect_df_sorted[effect_df_sorted["category"] != "no_meaningful_diff"].sort_values("mean_diff", ascending=False)
meaningful

In [ ]:
# %% [Layer 1 深入] P02 leading gene 组内对照效应量汇总 + 分类
effect_df_P02 = summarize_leading_gene_within_celltype(
    "P02", leading_genes, data, CONFIG,
    celltype="Proliferating T-ALL",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
)
effect_df_P02

In [ ]:
# %% [Layer 1 深入] 画图
plot_within_celltype_effect_summary(
    effect_df_P02, "P02", "Proliferating T-ALL",
    CONFIG["output_root"] + "layer1",
)

In [ ]:
# %% [Layer 1 深入] P04 leading gene 组内对照效应量汇总 + 分类
# P04 对 disease 的 AUC 只有0.33，对 Monocytes 的 AUC 高达0.99 ——
# 这里验证的是"肿瘤微环境对正常单核细胞的旁观者效应"，不是排除疾病混淆
effect_df_P04 = summarize_leading_gene_within_celltype(
    "P04", leading_genes, data, CONFIG,
    celltype="Monocytes",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
)
effect_df_P04

In [ ]:
# %% [Layer 1 深入] P04 画图
plot_within_celltype_effect_summary(
    effect_df_P04, "P04", "Monocytes",
    CONFIG["output_root"] + "layer1",
)

In [ ]:
# %% [Layer 1 深入] P07 leading gene 组内对照效应量汇总 + 分类
# P07 对 disease 的 AUC 只有0.56，对 Plasma cells 的 AUC 高达0.98
effect_df_P07 = summarize_leading_gene_within_celltype(
    "P07", leading_genes, data, CONFIG,
    celltype="Plasma cells",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
)
effect_df_P07

In [ ]:
# %% [Layer 1 深入] P07 画图
plot_within_celltype_effect_summary(
    effect_df_P07, "P07", "Plasma cells",
    CONFIG["output_root"] + "layer1",
)

In [ ]:
# %% [Layer 2] program usage 与各 contrast（疾病、细胞身份）的关联
assoc_results = compute_contrast_association(data, CONFIG, program_direction=program_direction)
assoc_results["disease"]


In [ ]:
# %% [Layer 2] 关联结果可视化：热图 + UMAP网格
plot_association_heatmap(assoc_results, CONFIG, CONFIG["output_root"] + "layer2")
plot_program_umap_grid(data, CONFIG, CONFIG["output_root"] + "layer2")


In [ ]:
# %% [Layer 2 深入] P04 program usage 组内对照（Monocytes内部，排除细胞组成混淆）
compare_within_celltype(
    data, CONFIG,
    celltype="Monocytes",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
    program="P04",
)

In [ ]:
# %% [Layer 2 深入] P07 program usage 组内对照（Plasma cells内部）
compare_within_celltype(
    data, CONFIG,
    celltype="Plasma cells",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
    program="P07",
)

In [ ]:
# %% [Layer 3] program核心基因集 / Jaccard相似度 / 专属marker
program_gene_sets = compute_program_gene_sets(data, CONFIG)
jaccard_mat = compute_jaccard_matrix(program_gene_sets["gene_sets"], data["programs"])
markers = select_program_markers(program_gene_sets, data, CONFIG, top_n=20, spec_min_z=0.5)

In [ ]:
# %% [Layer 3] 模块标签推断 + Jaccard热图
module_labels = resolve_module_labels(data, CONFIG)
plot_jaccard_heatmap(jaccard_mat, CONFIG["output_root"] + "layer3", module_labels=module_labels)


In [ ]:
# %% [Layer 3] 专属marker的功能富集
gene_lists = {f"{p}_markers": markers[p]["symbol"].tolist() for p in data["programs"]}
enrichment = run_enrichment(gene_lists, CONFIG)
plot_enrichment_bubble(enrichment, CONFIG["output_root"] + "layer3", top_n=6)


In [ ]:
# %% [Layer 3] 全部核心基因（非仅专属marker）的功能富集，作为对照
symbol_map = data.get("gene_symbol_map")

def _symbol(gene_id):
    if symbol_map is None:
        return gene_id
    if gene_id in symbol_map.index:
        return symbol_map.loc[gene_id]
    bare_id = gene_id.split(".")[0]
    if bare_id in symbol_map.index:
        return symbol_map.loc[bare_id]
    return gene_id

gene_lists_all = {}
for p in data["programs"]:
    ensg_list = program_gene_sets["gene_sets"][p]
    symbols = [_symbol(g) for g in ensg_list]
    n_before = len(symbols)
    symbols = [s for s in symbols if not str(s).startswith("ENSG")]
    gene_lists_all[f"{p}_all"] = symbols
    print(f"{p}: 核心基因{n_before}个 -> 映射到symbol {len(symbols)}个")

enrichment_all = run_enrichment(gene_lists_all, CONFIG)

plot_enrichment_bubble(
    enrichment_all,
    CONFIG["output_root"] + "layer3",
    top_n=6,
    filename="layer3_enrichment_bubble_all.png",
)

In [ ]:
# 接在 Part 3 之后，复用已有的 run_enrichment
supports_genes_P02 = effect_df_P02[effect_df_P02["category"] == "supports"].index.tolist()
opposes_genes_P02 = effect_df_P02[effect_df_P02["category"] == "opposes"].index.tolist()

gene_lists_split = {
    "P02_supports": supports_genes_P02,
    "P02_opposes": opposes_genes_P02,
}
enrichment_split = run_enrichment(gene_lists_split, CONFIG, min_genes=2)  # opposes只有2个基因，min_genes要调低
plot_enrichment_bubble(enrichment_split, CONFIG["output_root"] + "layer3", top_n=8, filename="layer3_enrichment_split_P02.png")

In [ ]:
supports_set_P02 = set(effect_df_P02[effect_df_P02["category"] == "supports"].index)
marker_symbols_P02 = set(markers["P02"]["symbol"])
overlap = supports_set_P02 & marker_symbols_P02
print(f"P02 supports基因: {supports_set_P02}")
print(f"P02 Layer3专属marker: {marker_symbols_P02}")
print(f"两者重合: {overlap} ({len(overlap)}/{len(supports_set_P02)})")

In [ ]:
gene_lists_retry = {"P02_supports": supports_genes_P02}
enrichment_supports_retry = run_enrichment(gene_lists_retry, CONFIG, min_genes=2)

In [ ]:
print(enrichment_split.keys())
print(len(enrichment_split.get("P02_supports", [])) if enrichment_split.get("P02_supports") is not None else "P02_supports缺失或为None")